# Chapter 13: The swap

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "13"
__file__ = os.path.abspath("run.py")

## The scene

A memo from headquarters: next quarter every analysis workstation in the force moves to a new
model family. Nobody can say which one yet. Varma reads it and asks one question. "Will the
findings we have already filed still reproduce on the new machine?"

You open the prompt folder. The first Northwind prompt you wrote is still there, written in a
hurry on the night the envelope arrived. It worked once. You never checked whether it works twice.

## The concept

Models change under you. Vendors retire versions, a new release follows instructions differently,
your own team moves to a cheaper or faster model, or (as here) headquarters decides for you. A
prompt that only works because of one model's habits breaks silently on the next one.

A **future-proof prompt** is one whose success does not depend on habits. It states a contract
that any capable instruction-following model can meet, and it is backed by a test that tells you
when the contract is broken.

```mermaid
flowchart LR
    subgraph Brittle["v1: tuned by feel"]
      B1["/no_think + 'world's greatest detective'<br/>+ 'a nice list'"] --> B2[Prose in whatever<br/>shape the model likes]
      B2 --> B3[Regex parser guessing<br/>that shape]
    end
    subgraph Robust["v2: written as a contract"]
      R1["task + rules + &lt;invoices&gt; delimiters"] --> R2[JSON forced by a<br/>schema in the request]
      R2 --> R3[json.loads + checks]
    end
    B3 -->|model swap| X[Silent breakage]
    R3 -->|model swap| E[Eval suite says<br/>pass or fail]
```

## How it works

The lesson keeps two versions of the same prompt as files, with front matter and a changelog:

```
lessons/13-future-proofing/prompts/
  v1.md          id, version 1, written_for: qwen3:1.7b, status: retired
  v2.md          id, version 2, written_for: any instruction-following model, status: current
  CHANGELOG.md   what changed, why, and how it was tested
```

The script runs both versions on two models. `qwen3:1.7b` is "today". `qwen3:0.6b` stands in for
"tomorrow", a model nobody tuned these prompts for. Code, not a person, parses each answer and
checks it against the three Northwind invoices.

![Brittle and contract prompts across a model swap](../../docs/diagrams/13-future-proofing.svg)

| Habit in v1 | Why it breaks | What v2 does instead |
|---|---|---|
| `/no_think` in the text | A Qwen3 control token. Other families read it as words. | Sends `think: false` as a request setting. |
| "World's greatest detective", "take a deep breath", "important for my career" | Folklore that helped some older models. Says nothing about the task. | States the task and rules plainly. |
| "A nice list with the amounts" | The model picks the shape, and the shape changes run to run. | JSON schema through structured output (chapter 08). |
| No delimiters | Evidence and instructions blur. | `<invoices>` tags. |
| No empty case | What should it say if nothing matches? | "If there are no Northwind invoices, return an empty list." |
| Parser written against one output | The parser is a guess about a habit. | `json.loads` plus field checks. |

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import json
import re
import string
from pathlib import Path

from lantern import case, chat, learn, load_config
from lantern.ui import (
    Table,
    chapter,
    console,
    evidence,
    finding,
    note,
    reply,
    save_shot,
    step,
    your_turn,
)

chapter("13", "The swap", """
A memo from headquarters: next quarter every analysis workstation in the force moves to a new
model family. Nobody can tell you which one yet. Varma reads it and asks one question. "Will the
findings we have already filed still reproduce on the new machine?" You open the prompt folder.
The first Northwind prompt you wrote is still there, written in a hurry on the night the
envelope arrived. It worked once. You never checked whether it works twice.
""")

HERE = Path(__file__).parent
EXPECTED = {"NW-0388": 62500.0, "NW-0402": 71200.0, "NW-0419": 94000.0}
SCHEMA = {
    "type": "array",
    "items": {"type": "object",
              "properties": {"invoice_id": {"type": "string"}, "total": {"type": "number"}},
              "required": ["invoice_id", "total"]},
}


def load_prompt(name: str) -> tuple[dict, str]:
    """A prompt file is YAML-style front matter plus a template body."""
    text = (HERE / "prompts" / name).read_text(encoding="utf-8")
    _, meta, body = text.split("---", 2)
    fields = dict(line.split(": ", 1) for line in meta.strip().splitlines())
    return fields, body.strip()


def parse_v1(text: str) -> dict[str, float]:
    """The parser written for v1's output: 'NW-0388 ... 62,500.00' on one line."""
    found = {}
    for line in text.splitlines():
        m = re.search(r"(NW-\d{4}).*?(\d{1,3}(?:,\d{3})+\.\d{2})", line)
        if m:
            found[m.group(1)] = float(m.group(2).replace(",", ""))
    return found


def parse_v2(text: str) -> dict[str, float]:
    try:
        return {row["invoice_id"]: float(row["total"]) for row in json.loads(text)}
    except (json.JSONDecodeError, KeyError, TypeError, ValueError):
        return {}


invoices = "\n\n".join(case.read(f"invoices/{n}.txt") for n in
                       ("NW-0388", "GTS-1182", "NW-0402", "HCS-0554", "NW-0419"))
evidence("five invoices from Reyes's mailbox (3 from Northwind)", invoices[:600] + "\n...")

meta1, v1 = load_prompt("v1.md")
meta2, v2 = load_prompt("v2.md")

### 1. The two versions

In [ ]:
step("1. The two versions")
reply(f"v{meta1['version']} ({meta1['status']}, written for {meta1['written_for']})", v1,
      style="red")
reply(f"v{meta2['version']} ({meta2['status']}, written for {meta2['written_for']})", v2,
      style="green")

cfg = load_config()["models"]
today, tomorrow = cfg["default"], cfg["small"]
note(f"'Today' is {today}. To simulate the swap, 'tomorrow' is {tomorrow}: a different model "
     "that nobody tuned these prompts for.")

p_v1 = learn.predict(
    f"v1 was written for {today} and its parser was written for v1's answers. Will that parser "
    f"read v1's answer on {today}?", ["yes", "no"])
p_v2 = learn.predict(
    f"On {tomorrow}, which nobody tuned these prompts for, how many of the three Northwind totals "
    "will v2 get exactly right?", ["0", "1", "2", "3"])

rows, replies, measured = [], {}, {}
for model in (today, tomorrow):
    for version, template, parser, fmt in (("v1", v1, parse_v1, None),
                                           ("v2", v2, parse_v2, SCHEMA)):
        step(f"2. {version} on {model}")
        r = chat(template.format(invoices=invoices), model=model, temperature=0,
                 max_tokens=400, format=fmt)
        reply(f"{version} on {model}", r.text[:900], meta=f"{r.output_tokens} tokens")
        got = parser(r.text)
        exact = sum(got.get(k) == v for k, v in EXPECTED.items())
        wrong = sorted(set(got) - set(EXPECTED))
        rows.append((model, version, len(got) > 0, f"{exact}/3", ", ".join(wrong) or "none"))
        who = "today" if model == today else "tomorrow"
        replies[version, who] = r.text
        measured[version, who] = {"parsed": len(got) > 0, "exact": exact, "extra": len(wrong)}
        if (version, who) == ("v1", "today"):
            learn.reveal(p_v1, answer=0 if got else 1,
                         why="A parser written against one answer is a guess about a habit. "
                         "\"A nice list\" leaves the shape to the model, and the shape moves "
                         "from run to run, even on the model the prompt was tuned for.")
        if (version, who) == ("v2", "tomorrow"):
            learn.reveal(p_v2, answer=exact,
                         why="v2 states the task, the rules and the output schema, so any capable "
                         "model can meet it. Whether the rows are the right ones is a separate "
                         "question: count the rows that should not be there.")

### 3. Did the findings survive the swap?

In [ ]:
step("3. Did the findings survive the swap?")
table = Table(title="Northwind payments: parsed by code, checked against the invoices")
for col in ("model", "prompt", "parsed?", "exact totals", "rows that should not be there"):
    table.add_column(col)
for model, version, parsed, exact, extra in rows:
    table.add_row(model, version, "yes" if parsed else "[red]no[/red]", exact, extra)
console.print(table)

note("v1's output is prose shaped by one model's habits. Its parser is a guess about those habits.")

note("v2 states a contract (schema, rules, delimiters) that any capable model can meet.")
note("A contract makes failures visible, not impossible: watch for extra rows on the small model.")
note("The changelog in prompts/CHANGELOG.md records why each line changed. Read it.")
note("What protects you on swap day is not the prompt. It is the eval that proves the prompt.")

### 4. The contract, checked by code

In [ ]:
step("4. The contract, checked by code")
MODEL_SPECIFIC = ("/no_think", "/think", "<|im_start|>", "<|im_end|>", "[INST]")
CONTRACT = {"fields": {"invoice_id": "string", "total": "number"},
            "id_field": "invoice_id", "id_pattern": r"^[A-Z]{2,4}-\d{4}$"}


def render(template: str, variables: dict[str, str]) -> str:
    """Fill a prompt template; refuse model-specific tokens and missing or unused variables."""
    for token in MODEL_SPECIFIC:
        if token in template:
            raise ValueError(f"model-specific token {token!r} in the template")
    names = {f for _, f, _, _ in string.Formatter().parse(template) if f}
    if names - set(variables) or set(variables) - names:
        raise ValueError(f"template needs {sorted(names)}, got {sorted(variables)}")
    return template.format(**variables)


def validate(text: str, contract: dict, *, source: str) -> list[str]:
    """What is wrong with a reply under the contract. [] means it meets it. practice/ builds it."""
    try:
        data = json.loads(text)
    except ValueError:
        return ["not JSON: prose, a code fence or a thinking block around the data"]
    if not isinstance(data, list):
        return ["not a JSON list"]
    kinds = {"string": str, "number": (int, float)}
    problems, seen = [], set()
    for i, row in enumerate(data):
        if not isinstance(row, dict) or set(row) != set(contract["fields"]):
            problems.append(f"row {i}: fields must be exactly {sorted(contract['fields'])}")
            continue
        for name, kind in contract["fields"].items():
            if isinstance(row[name], bool) or not isinstance(row[name], kinds[kind]):
                problems.append(f"row {i}: {name} must be a {kind}")
        rid = str(row[contract["id_field"]])
        if not re.match(contract["id_pattern"], rid):
            problems.append(f"row {i}: {rid!r} does not match {contract['id_pattern']}")
        elif rid not in source:
            problems.append(f"row {i}: {rid} is not in the material")
        if rid in seen:
            problems.append(f"row {i}: {rid} appears twice")
        seen.add(rid)
    return problems


p_meet = learn.predict("Of the four replies, how many meet the contract, checked by code?",
                       ["0", "1", "2", "3", "4"])
for label, template in (("v1", v1), ("v2", v2)):
    try:
        render(template, {"invoices": invoices})
        note(f"render() accepts {label}.")
    except ValueError as e:
        note(f"render() refuses {label}: {e}.")
contract = Table(title="Each reply against the contract")
for col in ("model", "prompt", "meets the contract?", "first problem"):
    contract.add_column(col)
passes = 0
for (version, who), text in replies.items():
    problems = validate(text, CONTRACT, source=invoices)
    passes += not problems
    measured[version, who]["contract"] = not problems
    contract.add_row(today if who == "today" else tomorrow, version,
                     "yes" if not problems else "[red]no[/red]", problems[0] if problems else "")
console.print(contract)
learn.reveal(p_meet, answer=passes,
             why="The contract checks shape, types and that every id exists in the material. It "
             "does not check that a row is a Northwind invoice: that is substance, and only the "
             "expected totals in step 3 test it.")

your_turn(
    "13",
    check=[
        "Compare the 'parsed?' column with the v1 replies above. Is v1's answer wrong, or only in "
        "a shape that `parse_v1` in run.py cannot read?",
        "Read the 'rows that should not be there' column. Open those invoices in case/invoices/ "
        "and check who issued each one.",
        "Open lessons/13-future-proofing/prompts/v1.md, v2.md and CHANGELOG.md side by side. "
        "Which v1 lines were removed, and what reason does the changelog give?",
    ],
    exercises=[
        "Copy prompts/v2.md to prompts/v3.md, set `version: 3`, add the rule \"vendor must be "
        "exactly 'Northwind Freight Ltd'\" and a v3 entry in prompts/CHANGELOG.md. In run.py, add "
        "a `vendor` string to `SCHEMA`, load v3 with `load_prompt` and add a v3 row to the loop.",
        "Run make lesson N=13 TIER=hosted (ANTHROPIC_API_KEY in .env). Both 'today' and "
        "'tomorrow' become the hosted model. Do v1 and v2 parse, and meet the contract?",
        "In run.py, change the regex in `parse_v1` so it also reads v1's bullet lines. Run it "
        "again tomorrow. Does your fix still match?",
        "Add /no_think to the start of the body in prompts/v2.md, after the front matter. Do the "
        "qwen3:1.7b rows change? What would another model family make of it?",
    ],
)

learn.record(
    v1_today_parsed=measured["v1", "today"]["parsed"],
    v1_today_exact=measured["v1", "today"]["exact"],
    v2_today_exact=measured["v2", "today"]["exact"],
    v2_today_extra=measured["v2", "today"]["extra"],
    v1_tomorrow_parsed=measured["v1", "tomorrow"]["parsed"],
    v1_tomorrow_exact=measured["v1", "tomorrow"]["exact"],
    v2_tomorrow_exact=measured["v2", "tomorrow"]["exact"],
    v2_tomorrow_extra=measured["v2", "tomorrow"]["extra"],
    v1_contract=measured["v1", "today"]["contract"] and measured["v1", "tomorrow"]["contract"],
    v2_contract=measured["v2", "today"]["contract"] and measured["v2", "tomorrow"]["contract"],
    contract_passes=passes,
)

v2_lines = []
for who, name in (("today", today), ("tomorrow", tomorrow)):
    m = measured["v2", who]
    if not m["contract"] and not m["extra"]:
        v2_lines.append(f"On {name} the contract-style prompt broke the contract itself.")
    elif m["extra"]:
        v2_lines.append(f"On {name} it found {m['exact']} of the three but also listed "
                        f"{m['extra']} supplier(s) that are not Northwind: right in shape, "
                        "wrong in substance.")
    else:
        v2_lines.append(f"On {name} it met the contract and listed only Northwind.")
v1_note = ("The hurried version produced prose its own parser could not read, even on the model "
           "it was written for." if not measured["v1", "today"]["parsed"] else
           "The hurried version happened to parse today; nothing in it promises it will tomorrow.")
finding(f"""
The three Northwind invoices total 227,700.00 KBD, each dated inside one of Malhotra's trips to
Port Sabran. {v1_note} {" ".join(v2_lines)} Only a test would have caught that.

Varma initials the changelog. "Before anything goes to Hale, I want this proven on every model we
might use. Not demonstrated. Proven."
""")
save_shot("13-future-proofing")

## The nuances

**1. Put settings in the request, not in the prompt text.** Thinking on or off, the output schema,
temperature and the stop sequences are API parameters. Putting them in the prose (`/no_think`,
"reply in JSON please") ties the prompt to one model family's training.

**2. Magic phrases age badly.** "Let's think step by step" and "take a deep breath" were measured
improvements on specific older models. Newer models are trained on instructions, not incantations.
They neither need the phrase nor reliably honour it. Chapter 09 shows where explicit reasoning
still pays.

**3. Specify the output, always.** "A nice list" is an invitation to vary. In this run, v1 produced
nested Markdown bullets on one model and a near copy of the invoices on the other. Neither matched
the parser. It "worked once" only in the sense that a person could read it.

**4. A contract makes failure visible, not impossible.** v2 parsed on both models, and both padded
it. On qwen3:1.7b it added GTS-1182, a Gulf Tug Services invoice, and still met the contract: the
shape was right and the content was wrong. On qwen3:0.6b it added rows for Gulf Tug Services and
Harbour Cleaning Services with the vendor's name in `invoice_id`, and the contract check caught
that: "'GULF TUG SERVICES' does not match ^[A-Z]{2,4}-\d{4}$". Structured output guarantees shape.
Only a test checks substance.

**5. Keep examples separate from instructions.** If you add few-shot examples (chapter 07), keep
them in their own clearly marked block. When a model swap changes how examples are weighted, you
can change the examples without touching the rules.

**6. Version prompts like code.** A prompt id, a version number, a status, the model it was tested
on and a changelog entry for each change. When Hale asks "which instructions produced this
finding?", the answer is a file and a version, not a memory.

**7. Pin the model as well as the prompt.** `qwen3:1.7b` can be re-pulled and silently change.
Record the model digest (Ollama shows it in `ollama list` and `/api/show`) alongside the prompt
version for anything you rely on.

**8. The safety net is the eval, not the prompt.** No wording survives every model. What survives is
a suite that runs every prompt on the new model before you switch and tells you what broke. That
is chapter 18.

## On a hosted model

```bash
make lesson N=13 TIER=hosted     # needs ANTHROPIC_API_KEY in .env and `uv sync --extra cloud`
make lesson N=13 TIER=hosted REPLAY=1   # once a run has been recorded, no key needed
```

On the hosted tier v2's schema goes through Claude's native structured outputs
(`output_config` with a strict JSON schema), and `think: false` becomes Claude's own thinking
setting, so v1's `/no_think` is just three words of text. `temperature` is not sent (current
Claude models reject sampling settings) and is reported on the console. Both "today" and
"tomorrow" resolve to claude-haiku-5-5 (`hosted.anthropic.models` in `config.yaml`), so the hosted
run is a swap to a different family, not between two hosted models.

| Measure | Local: qwen3:1.7b today, qwen3:0.6b tomorrow, Ollama, Apple M1 Pro 16 GB | Hosted: claude-haiku-5-5 |
|---|---|---|
| v1 read by its own parser, today / tomorrow | no / no | not run |
| v2 exact Northwind totals, today / tomorrow | 3/3 / 3/3 | not run |
| v2 rows that should not be there, today / tomorrow | 1 / 2 | not run |
| Replies that meet the contract, of 4 | 1 (v2 on qwen3:1.7b) | not run |

What the hosted run will test: whether v1's prose happens to fit its parser on a model it was
never tuned for (if it does, that is luck, not a contract), and whether the extra non-Northwind
rows were a small-model effect. That v2 meets the contract wherever the schema is enforced
natively should hold on both.

## Try this

1. Tighten v2 for the small model. Add a rule such as "vendor must be exactly 'Northwind Freight
   Ltd'" and a `vendor` field to the schema, then save the result as `v3.md` with a changelog
   entry. Does 0.6B stop padding?
2. Switch `provider` in `config.yaml` to a cloud model (with a key in `.env`) and run v1 and v2.
   Which version survives a completely different family?
3. Rewrite the v1 parser to cope with the bullet format. Run it again tomorrow. How long do you
   think that fix will last?
4. Add `/no_think` to the front of v2. Does it change anything on `qwen3:1.7b`? Would it on a
   model that is not Qwen?

## Master it

After this chapter you can tell a prompt that leans on one model's habits from one that states a
contract, keep settings in the request, version a prompt like code, and write the renderer and the
contract check that catch what a model swap breaks.

**Predict, then see.** `make lesson N=13` stops three times to ask what will happen: whether v1's
own parser reads v1 on the model it was written for, how many Northwind totals v2 gets exactly
right on the model nobody tuned it for, and how many of the four replies meet the contract. Each
answer is checked against your own run. Add `ARGS=--no-predict` to watch without stopping.

**Build it.** Write [`practice/exercise.py`](practice/exercise.py): `render`, which fills a
template and refuses model-specific tokens and missing or unused variables, and `validate`, which
lists everything wrong with a reply under the contract. Step 4 of the lesson runs both live.

```bash
make exercise N=13          # 8 offline tests on the chapter's own prompts, no model needed
make hint N=13              # one hint at a time
make solution N=13          # the reference, once you have tried
```

**Check yourself.** `make quiz N=13`: six questions, each with the reason behind the answer.

**Break it.** `make challenge N=13` checks your latest run against three targets: v2 on the
small model with exactly the three Northwind rows; v1 rescued without a schema on both models;
and v2 without structured output, failing the contract.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 13

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 13

In [ ]:
%run ../../scripts/learn.py hint 13

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 13

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

The three Northwind invoices total 227,700.00 KBD, and each is dated inside one of Malhotra's trips
to Port Sabran. The contract prompt reports all three on both models, in a form code can check.
The hurried version produced prose that its own parser could not read, even on the model it was
written for. On that model the contract was met in shape but broken in substance: it listed a Gulf
Tug Services invoice as Northwind's. On the smaller model the contract failed outright. Only a test
would have caught either.

Varma initials the changelog. "Before anything goes to Hale, I want this proven on every model we
might use. Not demonstrated. Proven."

**Next:** [Chapter 14: Ask the ledger](../14-tool-calling/README.md)